<a href="https://colab.research.google.com/github/Kei300/IA-Python-Libraries-Statistics/blob/main/InsideAirbnb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import altair as alt

In [ ]:
# Configuración estilo visual
sns.set_style("whitegrid")
plt.rcParams['figure.figsize']=(12,8)
# Deshabilitar el límite de 500 filas de Altair
alt.data_transformers.enable('default', max_rows = None)
print("Librerías importadas")

In [ ]:
#carga y limpieza de datos
url = "https://data.insideairbnb.com/mexico/df/mexico-city/2026-06-15/data/listings.csv.gz"
print("DataSet cargado")
try:
  df = pl.read_csv(url)
  print("Dataset cargado exitosamente c:")
except Exception as e:
  print(f"Error al cargar el dataset: {e}")

#Limpieza inicial
if not df.is_empty():
  df = df.with_columns(
      pl.col("price")
      .str.replace_all(r"\$","")
      .str.replace_all(r",","")
      .cast(pl.Float64, strict=False)
      .alias("price")
  ).drop_nulls(
      subset =["price"]
  ).filter(
      pl.col("price") > 0
  )
print("Limpieza inicial realizada")
display(df.head())

In [ ]:
# Análisis básico
print("1. ¿Cuál es el precio promedio por noche?")
avg_price = df.select(pl.col("price").mean()).item()
print(f"El precio promedio por noche es: {avg_price}")
print("2. ¿Cuáles son los tipos de alojamiento?")
room_types = df.group_by('room_type').agg(pl.len().alias('count')).sort('count', descending=True)
print(room_types)
print("3. ¿Cuáles son las 10 alcaldías con más alojamientos?")
top_10= df.group_by('neighbourhood_cleansed').agg(pl.len()
.alias('count')).sort('count', descending=True)#.head(10)
print(top_10)
print("¿Quiénes son los anfritiones con más propiedades")
top_hosts = df.group_by('host_name').agg(pl.len().alias('count')).sort('count', descending=True).head(10)
print(top_hosts)


In [ ]:
#visualizaciones
price_to_plot_df = df.filter(pl.col("price") < df.select(pl.col("price").quantile(0.95)).item()).to_pandas()
#usando altair
chart_hist = alt.Chart(price_to_plot_df).mark_bar().encode(
    alt.X("price:Q", bin=alt.Bin(maxbins=50), title= ("Precio por noche")),
    alt.Y("count():Q", title=("Frecuencia")),
    tooltip=(alt.Tooltip('count()', title="Frecuencia"), alt.Tooltip('price:Q', bin=True, title="Rango de precio"))
).properties(title="Distribución de precios por noche", width=700, height=400)

chart_hist.show()


In [ ]:
# gráficar to alcaldías
top_alcaldías_pd = top_10.to_pandas()
fig_hoods = px.bar(top_alcaldías_pd, y='neighbourhood_cleansed', x='count',
                   title='Alcaldías con más alojamientos', orientation='h',
                   color='count' ,color_continuous_scale=px.colors.sequential.Viridis)
fig_hoods.show()

In [ ]:
# Mapa
df_sample_for_plot = df.filter(pl.col('price') < df.select(pl.col('price').quantile(0.95)).item()).sample(n=5000, seed=42).to_pandas()
fig_map= px.scatter_mapbox(
    df_sample_for_plot,
    lat='latitude',
    lon='longitude',
    color='price',
    size='price',
    zoom=10,
    color_continuous_scale=px.colors.sequential.Viridis_r,
    size_max = 15,
    mapbox_style= "carto-positron",
    hover_name ="name",
    hover_data = {"neighbourhood_cleansed":True, "price":":$.2f"}
)
fig_map.update_layout(title="Mapa de alojamientos en la ciudad de México", legend_title_text="Precio (MXM")
fig_map.show()